# VM SB — cap 1.0 퍼즐: critic 온도를 처음부터 고정 `square_tent12i_cfix03` × 3 + `square_tent12i_cfix1` × 3, γ 진단 `square_tent12i_g099` × 3 (vm3_new 판형, 0→9 순서)

전용 G4 VM에서 **0부터 9까지 순서대로**. 1번은 런타임 재시작 → 2번부터. **코드 변경 있음**(`train.critic_alpha_fixed`, 9/16 — 6번 셀의 단위 테스트와 스모크가 통과해야 7번). 9 run(Square 150k, RAM ≈ 160 GB, 5~6시간), 9번이 끝나면 VM 반납. 기준 문서: `HANDOFF.md` §26. 짝 노트북: `colab/vm_square_a0.ipynb`(VM SA — α₀ 교란·배치 재현, 코드 변경 없음). 두 VM을 같은 밤에 띄운다.

**배경(§25)**: critic 타깃 보너스 상한 `min(α, cap)`은 되먹임을 끊지만(cap 0.3: α 20 → 1.6–2.0, Q_W 30k → 1.5k) 값에 **비단조**다 — cap 0.3은 127k 0.522 ± 0.068(대조군 0.485 동급), cap 1.0은 0.240 ± 0.025(모든 참조 대비 0/3; 대조군 대비 −0.245 ± 0.041). 값 말고 **물리는 시점**이 다르다: cap03은 online ≈ 10k(env ≈ 42k)에 Q_W ≈ 85에서 물리고, cap1은 online ≈ 24–26k에 Q_W ≈ 1,000에서 물린다. cap1의 곡선은 42k까지 다른 arm과 같다가 물린 뒤에 떨어진다(seed 평균 62k 0.553 → 97k 0.313 → 127k 0.240; 62k·97k 값은 9/15 밤 keepalive 상태 줄에서 파싱한 것으로 `results/2026-09-16/` 표에는 없음).

## 질문
- **Q2 "물림 충격" vs "수준 효과"**: 커지는 보너스에 맞춰 학습된 critic이 cap이 물리는 순간 "고정 보너스" 타깃으로 급변해 재적합하는 과도기에 Q_W가 일관성을 잃는다(물림 충격)면, **critic 온도를 min이 아니라 처음부터 상수로** 두면(물림 시점이 없음) 순수 수준 효과만 남는다. 구현(`o2o_utils.critic_ent_coef`): `train.critic_alpha_fixed=c` → 타깃 `r + γ(Q̄ − c·log π′)`(첫 업데이트부터), actor 손실 `α·log π − Q_W`는 auto-α 그대로; cap과 상호 배타(`critic_temperature_settings`가 거부); fingerprint 포함; train_log `critic_ent_coef` = c.
  - `square_tent12i_cfix1_s{1,2,3}`: `train.ent_coef=auto_0.3 train.target_ent=12 train.critic_alpha_fixed=1.0`. 고정 1.0이 잘 되면 cap1의 실패는 물림 충격(처방: 부드러운 cap 또는 처음부터 물리는 cap); 고정 1.0도 안 되면 수준 효과(1.0은 그냥 과함).
  - `square_tent12i_cfix03_s{1,2,3}`: `… train.critic_alpha_fixed=0.3` — sanity arm. cap03은 online 10k부터 92% 행에서 물리므로 cfix03 ≈ cap03이어야 한다(min과 고정이 같은 것).
- **Q3 γ 진단**: 되먹임 고리(보너스 → Q_W ↑ → actor Q 기울기 ↑ → 엔트로피 12 유지에 α ↑ → 보너스 ↑)의 이득은 H/(1−γ)에 비례한다 — Can(γ 0.99) 100배, Square(γ 0.999) 1,000배. `square_tent12i_g099_s{1,2,3}` = `… train.discount=0.99`(코드 변경 없음; `discount`가 fingerprint에 들어가므로 0.999 checkpoint는 resume 거부). 예측: α가 2 근처에 머문다. 과제 목적함수를 바꾸므로 **처방이 아니라 기제 확인용**.

## 사전 판정 (정의는 `results/2026-09-08/README.md` §2; 후반 env 127,136·200 ep; seed-matched = 같은 seed 번호끼리, 9/15 cap arm과 짝; §23 규칙 paired df=2 |t| > 4.3 ∧ 3/3)
- **Q2 주 지표 = seed-matched `cfix1 − cap1`(s1–3) 127k**(cap1 0.235/0.285/0.200, 평균 0.240; 9/15 배치와의 교차 배치 대비). (a) **평균 차 ≥ +0.15 ∧ 3/3 → "물림 충격 지지"**; (b) **평균 차 ≤ +0.075(음수 포함) → "수준 효과(시사)"**(고정 1.0도 cap 1.0보다 낫지 않음 → 1.0은 과함, 상한은 0.3 쪽; n=3 동등성이라 '시사'); (c) 그 외(0.075 < 차 < 0.15, 또는 ≥ 0.15인데 3/3 아님) → 미결. late(102k·127k 평균) 차도 같은 (a)/(b)/(c)로 분류해 함께 보고하고, **127k가 (a)인데 late가 (b), 또는 127k가 (b)인데 late가 (a)면 미결로 강등**. 보조((a)일 때만): cfix1 127k ≥ 0.472(baseline s1–3 = §24 판정표의 '후반 해결' 문턱; 대조군 tent12i 0.485는 그 SE 안)이면 cap1 손실 전부가 물림 충격. **Q1a 의존**: Q1a = 배치 효과면 (a)의 문턱을 +0.15 + |tent12r − tent12 127k 차|로 올리고 (b)는 미결로 강등; Q1a = 미결이면 (b)만 미결로 강등((a) 문턱 그대로); Q1a = 배치 차이 없음이면 그대로.
- **Q2 sanity = `cfix03 − cap03`(s1–3) late(102k·127k 평균; cap03 127k 0.430/0.655/0.480은 SD 0.118이라 단일점은 잡음)**: |평균 차| ≤ 0.075 예상(cap03은 online 10k부터 92% 행에서 물림 → 거의 같은 연산). 이 항목은 어느 결과든 **기록만**(판정 아님); 3/3 ∧ |t| > 4.3이면 cap03의 안 물린 첫 10k가 결과를 바꾼 것으로 적는다.
- **Q3 = online 110k 5k bin(110,000–114,999)의 actor α seed 평균**(train_log `ent_coef`; 9/16 `diag_bins.csv`의 같은 bin: tent12i 18.2, cap03 1.71, cap1 3.58; 90k bin은 13.7/1.56/3.16): **< 3 → "γ가 폭주 원천" 지지**; **≥ 10 → 기각**; 사이 → 미결. Q_W 예측: tent12i(29.5k)의 ~1/10. 성능(42k·127k)은 목적함수가 달라 **판정하지 않고 기술만**(eval의 mc_return도 γ 0.99 눈금).
- 진단 예측: cfix arm의 `critic_ent_coef` = c 전 행(구성상; 6번 스모크가 잰다), actor α는 cap arm과 비슷(cap03 1.6–2.0, cap1 3.6–3.8), Q_W는 cap1 후반 수준(≈4k)에 급변 없이 도달. cfix1의 초반(온도 1.0으로 시작 → 보너스 ≈ 12/step)은 42k에서 cap1 0.343·대조군 0.390과 기술만.
- 쓰면 안 되는 문장: §25·README §5 그대로. 추가: "cap1의 실패는 물림 충격/수준 효과"(이 실험이 그 검정 — 결과 전엔 가설), "γ 0.99가 Square 처방"(목적함수 변경), "MDAC과 방향 일치"(MDAC의 '공격적'은 자주 물리는 bound).


## 0. Drive 마운트와 GPU 확인


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Conda 설치 — 실행하면 런타임 자동 재시작


In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. 재시작 후 Drive 재마운트


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print(PROJ)

## 3. 최신 o2o 저장소 준비 — HEAD가 `critic_alpha_fixed` 커밋(9/16) 이상이어야 한다


In [ ]:
%%bash
set -e
git config --global url."https://github.com/".insteadOf "git@github.com:"
if [ -d /content/dsrl/.git ]; then
  git -C /content/dsrl checkout o2o
  git -C /content/dsrl pull --ff-only origin o2o
else
  test ! -e /content/dsrl || { echo '/content/dsrl exists but is not a git checkout'; exit 2; }
  git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git /content/dsrl
fi
git -C /content/dsrl submodule sync --recursive
git -C /content/dsrl submodule update --init --recursive
echo -n 'HEAD: '; git -C /content/dsrl rev-parse --short HEAD
grep -q 'critic_alpha_fixed' /content/dsrl/o2o_utils.py || { echo 'o2o_utils.py has no critic_alpha_fixed: the 9/16 commit is not on origin/o2o yet'; exit 2; }


## 4. 캐시에서 conda 환경 복원


In [ ]:
%%bash
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache/dsrl_env.tar.gz
test -s "$CACHE" || { echo "missing $CACHE"; exit 2; }
mkdir -p /usr/local/envs
rm -rf /usr/local/envs/dsrl
tar -xzf "$CACHE" -C /usr/local/envs
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
assert torch.cuda.is_available(), 'GPU runtime required'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))
PY
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 5. Square 정책·정규화 복원과 환경 패치 (스모크는 Can 정책도 쓰므로 Can 자산도 복원)


In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
RUNTIME=/content/dsrl/dppo/log
DRIVE=$PROJ/dppo_log
mkdir -p "$RUNTIME"
test -d "$DRIVE" || { echo "missing $DRIVE"; exit 2; }
cp -r "$DRIVE"/. "$RUNTIME"/
CAN_CKPT=robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt
CAN_NORM=robomimic/can/normalization.npz
for REL in "$CAN_CKPT" "$CAN_NORM"; do
  SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$REL" -print -quit)
  test -n "$SRC" || { echo "missing Can asset for the smoke: $REL"; exit 2; }
  mkdir -p "$RUNTIME/$(dirname "$REL")"; [ "$SRC" = "$RUNTIME/$REL" ] || cp -f "$SRC" "$RUNTIME/$REL"
done
CKPT_REL=robomimic-pretrain/square/square_pre_diffusion_mlp_ta4_td100_ddim-100steps/2025-04-11_19-13-26_44/checkpoint/state_3000.pt
NORM_REL=robomimic/square/normalization.npz
CKPT_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$CKPT_REL" -print -quit)
NORM_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$NORM_REL" -print -quit)
test -n "$CKPT_SRC" -a -n "$NORM_SRC" || { echo 'Square policy assets missing'; exit 2; }
mkdir -p "$RUNTIME/$(dirname "$CKPT_REL")" "$RUNTIME/$(dirname "$NORM_REL")"
[ "$CKPT_SRC" = "$RUNTIME/$CKPT_REL" ] || cp -f "$CKPT_SRC" "$RUNTIME/$CKPT_REL"
[ "$NORM_SRC" = "$RUNTIME/$NORM_REL" ] || cp -f "$NORM_SRC" "$RUNTIME/$NORM_REL"
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS
python /content/dsrl/colab/patch_env.py
ls -lh "$RUNTIME/$CKPT_REL" "$RUNTIME/$NORM_REL"

## 6. 사전검사 — 단위 테스트 3종 + 고정 온도 스모크(Can, 작은 망, 약 5분) + 이번 exp_id 9개·비교군·RAM

스모크: 고정 actor α 1.0으로 1,200 env step을 두 번 돌려 `critic_ent_coef` 열이 `critic_alpha_fixed=2.0`이면 전부 2.0(= min(α, ·)이 아니라 상수임을 증명), −1이면 `ent_coef`와 같은지 확인한다(둘 중 하나라도 어긋나면 셀이 실패하고 7번으로 가지 않는다). 스모크 폴더는 확인 뒤 지운다.


In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
echo '== 단위 테스트 (torch 불필요): fixed 기본값·fingerprint·cap 배타, discount fingerprint, resume 상태, launch 문자열'
python scripts/test_offline_mix.py 2>&1 | tail -n 2; test ${PIPESTATUS[0]} -eq 0 || { echo 'test_offline_mix FAILED'; exit 2; }
python scripts/test_resume_state.py 2>&1 | tail -n 2; test ${PIPESTATUS[0]} -eq 0 || { echo 'test_resume_state FAILED'; exit 2; }
python scripts/test_notebook_launches.py 2>&1 | tail -n 2; test ${PIPESTATUS[0]} -eq 0 || { echo 'test_notebook_launches FAILED'; exit 2; }
echo '== 고정 온도 스모크 (Can, 1,200 env step x 2)'
CFG_CAN='--config-path=cfg/robomimic --config-name=dsrl_can.yaml'
SMOKE="log_dir=$PROJ/logs env.n_envs=1 env.n_eval_envs=1 num_evals=1 eval_schedule.every_env_early=400 eval_schedule.early_until_env=100000 eval_schedule.num_evals_early=1 ckpt_every_env_steps=400 train.init_rollout_steps=50 train.utd=1 train.noise_critic_grad_steps=1 train.batch_size=32 train.layer_size=256 train.num_layers=2 train.buffer_size=20000 train.total_env_steps=1200 resume=False train.ent_coef=1.0"
for FIXED in 2.0 -1; do
  rm -rf "$PROJ/logs/smoke_cfix$FIXED"
  python train_dsrl.py $CFG_CAN exp_id=smoke_cfix$FIXED $SMOKE train.critic_alpha_fixed=$FIXED 2>&1 | grep '\[done\]\|Error\|Traceback' | tail -n 2; test ${PIPESTATUS[0]} -eq 0 || { echo "smoke fixed=$FIXED FAILED"; exit 2; }
  python - "$PROJ/logs/smoke_cfix$FIXED/train_log.csv" "$FIXED" <<'PY'
import csv, sys
path, fixed = sys.argv[1], float(sys.argv[2])
rows = list(csv.DictReader(open(path)))
assert rows and 'critic_ent_coef' in rows[0], 'critic_ent_coef column missing: ' + path
a = [float(r['ent_coef']) for r in rows]
c = [float(r['critic_ent_coef']) for r in rows]
assert all(abs(x - 1.0) < 1e-6 for x in a), 'the actor alpha must stay at the fixed 1.0'
want = [fixed if fixed > 0 else x for x in a]
assert all(abs(x - y) < 1e-6 for x, y in zip(c, want)), f'critic_ent_coef {c[:3]} != expected {want[:3]}'
print(f'OK fixed={fixed}: {len(rows)} rows, critic_ent_coef {c[0]:.3f} (ent_coef {a[0]:.3f})')
PY
  rm -rf "$PROJ/logs/smoke_cfix$FIXED"
done
echo '== cap과 fixed를 같이 주면 거부되는지 (train_dsrl.py가 env를 만들기 전에 거부)'
rm -rf "$PROJ/logs/smoke_cfix_both"
python train_dsrl.py $CFG_CAN exp_id=smoke_cfix_both $SMOKE train.critic_alpha_cap=0.3 train.critic_alpha_fixed=1.0 > /tmp/smoke_both.out 2>&1 && { echo 'cap+fixed was accepted (must be refused)'; exit 2; } || true
grep -q 'mutually exclusive' /tmp/smoke_both.out || { echo 'cap+fixed failed for another reason:'; tail -n 5 /tmp/smoke_both.out; exit 2; }
echo 'OK cap+fixed refused'; rm -rf "$PROJ/logs/smoke_cfix_both"
echo '== exp_id 9개 (있으면 같은 명령이 checkpoint resume; 온도 규칙·γ가 다른 checkpoint는 fingerprint가 거부)'
for E in square_tent12i_cfix03_s1 square_tent12i_cfix03_s2 square_tent12i_cfix03_s3 square_tent12i_cfix1_s1 square_tent12i_cfix1_s2 square_tent12i_cfix1_s3 square_tent12i_g099_s1 square_tent12i_g099_s2 square_tent12i_g099_s3; do
  if [ -f "$PROJ/logs/$E.out" ]; then echo "$E: 이미 있음 -> $(grep '\[done\]\|\[eval\]' $PROJ/logs/$E.out | tail -n 1 | cut -c1-80)"; else echo "$E: 새로 시작"; fi
done
echo '== 9/15 비교군 (tent12i 3, cap03 3, cap1 3 이어야 함)'
for G in square_tent12i square_tent12i_cap03 square_tent12i_cap1 square_tent12 square_tent12_hq; do echo -n "비교군 $G: "; ls -d $PROJ/logs/${G}_s*/ 2>/dev/null | wc -l; done
echo "processes: $(pgrep -fc '[t]rain_dsrl.py' || true)"; free -g | head -2; df -h /content | tail -n 1


## 7. 9개 시작 — `square_tent12i_cfix03` × 3 + `square_tent12i_cfix1` × 3 + `square_tent12i_g099` × 3 (150k, 5k 격자). 6번이 실패했으면 띄우지 않는다


In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
git pull --ff-only origin o2o
grep -q 'critic_alpha_fixed' o2o_utils.py || { echo 'critic_alpha_fixed missing from o2o_utils.py'; exit 2; }
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p "$PROJ/logs"
CFG='--config-path=cfg/robomimic --config-name=dsrl_square.yaml'
COMMON="variant=baseline log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=none load_offline_data=False"
T12I="train.ent_coef=auto_0.3 train.target_ent=12"
launch () {
  EXP=$1; shift
  if pgrep -af '[t]rain_dsrl.py' | grep -Fq "exp_id=$EXP "; then echo "already running: $EXP"; return; fi
  nohup python train_dsrl.py $CFG exp_id=$EXP "$@" $COMMON > "$PROJ/logs/$EXP.out" 2>&1 &
  echo "started $EXP (pid $!)"
}
for S in 1 2 3; do
  launch square_tent12i_cfix03_s$S seed=$S $T12I train.critic_alpha_fixed=0.3
  launch square_tent12i_cfix1_s$S  seed=$S $T12I train.critic_alpha_fixed=1.0
  launch square_tent12i_g099_s$S   seed=$S $T12I train.discount=0.99
done


## 8. 3분 후 자동 확인 — 9개 running, ERR 없음, 인자에 `train.critic_alpha_fixed=0.3`/`1.0`(cfix)·`train.discount=0.99`(g099)와 `auto_0.3 … target_ent=12`. 30분 뒤 다시 돌리면 train_log의 `ent_coef`(actor α)·`critic_ent_coef`(cfix arm은 처음부터 = c)·qw_mean도 찍힌다


In [ ]:
%%bash
sleep 180
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python scripts/inspect_runs.py --proj "$PROJ" --only square_tent12i_cfix03_s,square_tent12i_cfix1_s,square_tent12i_g099_s
for E in square_tent12i_cfix03_s1 square_tent12i_cfix03_s2 square_tent12i_cfix03_s3 square_tent12i_cfix1_s1 square_tent12i_cfix1_s2 square_tent12i_cfix1_s3 square_tent12i_g099_s1 square_tent12i_g099_s2 square_tent12i_g099_s3; do
  echo "== $E: $(grep '\[budget\]\|\[eval\]\|Traceback\|Error' "$PROJ/logs/$E.out" | tail -n 2 | tr '\n' ' ' | cut -c1-160)"
  T=$PROJ/logs/$E/train_log.csv
  [ -f "$T" ] && awk -F, 'NR==1{for(i=1;i<=NF;i++)c[$i]=i} END{printf "   train_log last: env_steps=%s ent_coef=%s critic_ent_coef=%s logp_mean=%s qw_mean=%s mu=%s\n", $c["env_steps"], $c["ent_coef"], $c["critic_ent_coef"], $c["logp_mean"], $c["qw_mean"], $c["mu_absmean"]}' "$T"
done
echo '== processes (인자 확인)'; pgrep -af '[t]rain_dsrl.py' | sed 's/.*exp_id=/exp_id=/' | cut -c1-200 || true
free -g | head -2


## 9. Keepalive — 마지막 프로세스가 끝나면 자동 반납

8번에서 오류가 없을 때만 실행하고 이 셀을 계속 실행 상태로 둡니다.


In [ ]:
import subprocess, time
from pathlib import Path
EXPECTED = [f'square_tent12i_{kind}_s{s}' for kind in ('cfix03', 'cfix1', 'g099') for s in (1, 2, 3)]
LOGS = Path('/content/drive/MyDrive/dsrl_project/logs')
assert LOGS.is_dir(), 'Drive가 이 런타임에 마운트돼 있지 않음 — 0번(또는 2번) 셀 먼저'

def running():
    out = subprocess.run(['ps', '-eo', 'pid,args'], capture_output=True, text=True).stdout
    return [line.strip() for line in out.splitlines() if 'train_dsrl.py' in line and any(f'exp_id={e} ' in line + ' ' for e in EXPECTED)]

def last_event(exp):
    # 마지막 2,000줄 안에서 찾는다: SB3 verbose 표(≈30줄/1,600 step)가 102k→127k의 25k 구간에 500줄을 넘겨
    # 9/16에는 '[eval]'이 창 밖으로 밀려 'starting'으로 보였다(프로세스는 정상이었음).
    path = LOGS / f'{exp}.out'
    if not path.exists(): return 'NO .out'
    lines = path.read_text(errors='replace').splitlines()[-2000:]
    for line in reversed(lines):
        if any(x in line for x in ('[eval]', '[done]', 'Traceback', 'Error')): return line[:100]
    return 'no [eval] yet'

seen_running = False
while True:
    procs = running()
    events = {e: last_event(e) for e in EXPECTED}
    print(time.strftime('%H:%M'), f'running {len(procs)}/{len(EXPECTED)}', '|', ' | '.join(f'{e}: {v}' for e, v in events.items()), flush=True)
    if procs:
        seen_running = True
    elif seen_running or all(v.startswith('[done]') for v in events.values()):
        print('all VM SB runs stopped -> unassigning', flush=True)
        from google.colab import runtime
        runtime.unassign()
        break
    else:
        print('이 런타임에 실행 중인 run이 없고 [done]도 아님 -> 7번 셀이 안 돌았거나 다른 런타임입니다. 반납하지 않고 종료.', flush=True)
        break
    time.sleep(600)


## 10. 결과 zip (끝난 뒤, CPU 런타임 + 0번 Drive 마운트만으로 됨). 로컬에서는 **새 폴더**(예: `logs/bundle_<날짜>/`)에 풀고 — 옛 번들이 섞인 `~/Downloads/logs`는 쓰지 않는다 —
`python scripts/plot_results.py --logs <새폴더>/logs --out results/<날짜>/square_a0_cfix --axes "square_a0=square_baseline,square_tent12,square_tent12r,square_tent12_hq,square_tent12i,square_tent12i_cap03;square_cfix=square_tent12,square_tent12i,square_tent12i_cap03,square_tent12i_cap1,square_tent12i_cfix03,square_tent12i_cfix1,square_tent12i_g099"`

`.out` 파일은 번들에 들어가지 않는다(Drive에만). 판정은 `HANDOFF.md` §26의 사전 등록 그대로: n=3 대비는 §23 규칙(paired df=2, |t| > 4.3 ∧ 3/3), n=5 대비는 df=4 |t| > 2.78 ∧ ≥ 4/5.


In [ ]:
%%bash
cd /content/drive/MyDrive/dsrl_project
rm -f csv_bundle.zip
zip -qr csv_bundle.zip logs -i "logs/*/eval_log.csv" "logs/*/train_log.csv" "logs/*.csv" "logs/pretrain/*_log.csv"
ls -lh csv_bundle.zip